In [ ]:
import torch.utils.data
from hydra import initialize, compose
from omegaconf import DictConfig
import os

from sentence_transformers.util import similarity

from src.data.dataloader import data_loader
from src.data.dataset_aprdb import AprdbDataset

import src.config
from src.detector.cristal_anchor.fetch_academic_paper import extract_text_from_pdf

with initialize(version_base=None, config_path="../conf"):
    cfg: DictConfig = compose(config_name="config", overrides=["+experiment/dataset/aprbd@dataset=aprdb_tt_test_all",
                                                               "experiment/data_loader=1_4_true"])
EXPERIENCE = "reviews_qwen3-14b_iclr2019_ibm"
os.environ['CUDA_LAUNCH_BLOCKING'] = "1"

print(cfg)

In [ ]:
ds_test = AprdbDataset(config=cfg.dataset)
len(ds_test)

### getting dataframe conversion

In [ ]:
import pandas as pd

rows = []
for i in range(len(ds_test)):
    item = ds_test[i]
    rows.append(item)
df = pd.DataFrame(rows)
df.dropna(subset=['forum_id'], inplace=True)
df

In [ ]:
year = "2019"
conference = "ICLR"
generator = "qwen"
mgt_df = df[(df.label == 1) & (df['conference'] == conference) & (df['year'] == year) & (df['generator'] == generator)]

hwt_df = df[(df.label == 0) & (df.conference == conference) & (df.year == year)]
mgt_df

sampled = pd.concat([
    mgt_df.sample(n=25, random_state=42),
    hwt_df.sample(n=25, random_state=42)
])

In [ ]:
from datasets import Dataset as HFDataset

dataloader = data_loader(
    cfg.experiment.data_loader,
    HFDataset.from_pandas(sampled, preserve_index=False),
)

In [ ]:
random_batch = next(iter(dataloader))
batch_id = random_batch['review-id']
print(random_batch)
print(batch_id, random_batch['content'][0])

# reverse Prompt Engineering

In [ ]:
txt = random_batch['content']
from src.detector.cristal_anchor.fetch_academic_paper import extract_text_from_pdf

paper = extract_text_from_pdf(random_batch['pdf_path'][0])

In [ ]:
def build_system_propmt(reviewer_guideline, review_template): return f"""
You are an AI researcher reviewing a paper submitted to a prestigious AI research conference. You will be provided with the manuscript text, the conference's reviewer guidelines, and the decision for the paper. Your objective is to thoroughly evaluate the paper, adhering to the provided guidelines, and return a detailed assessment that supports the given decision using the specified response template. Ensure your evaluation is objective, comprehensive, and aligned with the conference standards. {reviewer_guideline} {review_template}
    """

In [ ]:
ICLR_2019_GUIDELINES = """Publication standards. It’s important to maintain the high quality of ICLR papers. At the same time, we must be cognizant of the fact that different reviewers, authors, and even different areas will have different standards about what constitutes a high-quality paper. Reviewers are encouraged to use their best judgement, but also ask themselves the following questions:

Does the paper present substantively new ideas or explore an underexplored or highly novel question? Papers that take risks and study a less explored area are likely to have less polished results, papers that study a highly explored topic are likely to have more polished results. This phenomenon often results in reviewers excessively penalizing papers that explore underexplored topics, and it is worth accounting for this.

At the same time, papers that present state-of-the-art results on well-studied problems should be given consideration: addressing problems that are clearly of interest to a large segment of the community is important, when the results substantively advance the state of the art.

Ask yourself: will a substantial fraction of the ICLR attendees be interested in reading this paper?

Another question to ask yourself: would I send this paper to one of my colleagues to read? This is often a good indication that the paper is worth accepting, though not the only indication.

At the same time, it is critical to maintain a high standard in terms of scientific rigor: if you believe that a paper has flaws in terms of its evaluation or validation, proofs, or other parts of the discussion, it is critical to point this out to the authors. The authors will have an opportunity to address these concerns, and the iterative process of improving papers after reviewer feedback is important for ensuring the highest quality of ICLR papers."""

In [ ]:
ICLR_TEMPLATE = """The format should be the format review for an ICLR review"""

In [ ]:
def build_user_prompt(text):
    return f"""
Here is the paper you are asked to review. Write a well-justified review of this paper that aligns with a ``` {text} ```
"""

In [ ]:
import re

from openai import OpenAI

VLLM_API_BASE = "http://127.0.0.1:8000/v1"
MODEL_NAME = "qwen3-14b"
_client = OpenAI(base_url=VLLM_API_BASE, api_key="EMPTY")


def chat(messages, *, system_prompt, user_prompt, enable_thinking=False, temperature=0.7, top_p=0.95,
         top_k=20, max_tokens=2048):
    if enable_thinking and temperature > 0.6:
        temperature = 0.6
    resp = _client.chat.completions.create(
        model=MODEL_NAME,
        messages=messages,
        temperature=temperature,
        top_p=top_p,
        max_tokens=max_tokens,
        extra_body={
            "top_k": top_k,
            "chat_template_kwargs": {"enable_thinking": enable_thinking},
        },
    )
    msg = resp.choices[0].message
    return {
        "content": (msg.content or "").strip(),
        "reasoning": getattr(msg, "reasoning_content", None),
        "finish_reason": resp.choices[0].finish_reason,
        "usage": dict(resp.usage),
        "system_prompt": system_prompt,
        "user_prompt": user_prompt,
    }


def _extract_tag(text: str, tag: str, required: bool = True) -> str | None:
    """Extrait le contenu d'un tag. Si non requis et absent, renvoie None."""
    m = re.search(rf"<{tag}>(.*?)</{tag}>", text, flags=re.DOTALL)
    if not m:
        # Fallback : tag ouvert mais pas refermé (sortie tronquée)
        m_open = re.search(rf"<{tag}>(.*)", text, flags=re.DOTALL)
        if m_open and not required:
            return m_open.group(1).strip()
        if required:
            raise ValueError(f"Required tag <{tag}> not found in output")
        return None
    return m.group(1).strip()


def generate_review(
        paper: str,
        enable_thinking: bool = False,
        temperature: float = 0.7,
        max_tokens: int = 4096) -> dict:
    """Étape 2 : applique le prompt reconstruit au papier cible."""
    system_prompt = build_system_propmt(ICLR_2019_GUIDELINES, ICLR_TEMPLATE)
    user_prompt = build_user_prompt(paper)
    return chat(
        [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        system_prompt = system_prompt,
        user_prompt = user_prompt,
        enable_thinking=enable_thinking,
        temperature=temperature,
        max_tokens=max_tokens,
    )


def review_pipeline(paper_text: str) -> dict:
    max_token = 4093

    generated = generate_review(paper_text, max_tokens=max_token, enable_thinking=False)
    return {
        "generated_review": generated["content"],
        "generated_reasoning": generated["reasoning"],
        "user_prompt": generated["user_prompt"],
        "system_prompt": generated["system_prompt"],
        "usage": {
            "generate": generated["usage"],
        },
    }



In [ ]:
# === Test ===
r = random_batch["content"][0] if isinstance(random_batch["content"], list) else random_batch["content"]
p = extract_text_from_pdf(random_batch["pdf_path"][0])
result = review_pipeline(p)



In [ ]:

print("\n=== GENERATED REVIEW ===")
print(result["generated_review"], len(result["generated_review"]))
print("\n=== REASONING ===")
print("\n=== GENERATED REVIEW ===")
print("\n=== USAGE ===")
print(result["usage"])
result_review = result['generated_review']

In [ ]:
print(str(random_batch['content'][0]), f"\n----------------- END -------------------\n", str(result_review))
print(len(random_batch['content'][0].split()), len(str(result_review).split()))

In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

In [ ]:
## test embedding
embedded_target = embedding_model.encode(random_batch["content"][0])
embedded_generated_review = embedding_model.encode(result_review)

similarity = embedding_model.similarity(embedded_target, embedded_generated_review)
print(similarity)

In [ ]:
from datasets import tqdm

label = []
target = []
infos = []
for batch in tqdm(dataloader):
    try:
        r = batch["content"][0] if isinstance(batch["content"], list) else batch["content"]
        p = extract_text_from_pdf(batch["pdf_path"][0])

        result = review_pipeline(p)
        result_review = result['generated_review']

        embedded_target = embedding_model.encode(random_batch["content"][0])
        embedded_generated_review = embedding_model.encode(result_review)

        similarity = embedding_model.similarity(embedded_target, embedded_generated_review)

        infos.append({
            "label": batch['label'],
            "target_review": r,
            "generated_review": result_review,
            "similarity": similarity,
            "usage": result["usage"],
            "paper_id": batch['forum_id'],
            "review-id": batch['review-id']

        })

        label.append(batch['label'])
        target.append(similarity)
    except:
        print("error")
        continue


In [ ]:
label

In [ ]:
import json


def to_serializable(x):
    if isinstance(x, torch.Tensor):
        return x.detach().cpu().tolist()
    return x


infos_clean = [{k: to_serializable(v) for k, v in d.items()} for d in infos]
with open("reviews_qwen3-14b_iclr2019_ibm.json", "w", encoding="utf-8") as f:
    json.dump(infos_clean, f, ensure_ascii=False, indent=2)


In [ ]:
print(infos_clean)

In [ ]:
label = torch.stack(label)
target = torch.cat(target).flatten()
label

In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc

# label : ground truth binaire (0/1) ; target : score continu du détecteur
fpr, tpr, thresholds = roc_curve(label, target)
roc_auc = auc(fpr, tpr)

# Seuil maximisant TPR sous contrainte FPR <= 5%
mask = fpr <= 0.05
idx = np.argmax(tpr[mask])  # meilleur TPR dans la zone admissible
thr_at_5fpr = thresholds[mask][idx]
tpr_at_5fpr = tpr[mask][idx]
fpr_at_5fpr = fpr[mask][idx]

plt.figure(figsize=(6, 6))
plt.plot(fpr, tpr, label=f"ROC (AUC = {roc_auc:.4f})")
plt.plot([0, 1], [0, 1], "k--", alpha=0.4)
plt.axvline(0.05, color="red", linestyle=":", label="FPR = 5%")
plt.scatter([fpr_at_5fpr], [tpr_at_5fpr], color="red", zorder=5,
            label=f"TPR@5%FPR = {tpr_at_5fpr:.3f} (thr={thr_at_5fpr:.4f})")
plt.xlabel("FPR");
plt.ylabel("TPR")
plt.title("ROC curve");
plt.legend(loc="lower right")
plt.grid(alpha=0.3);
plt.tight_layout();
plt.show()

print(f"AUROC          : {roc_auc:.4f}")
print(f"TPR @ 5% FPR   : {tpr_at_5fpr:.4f}")
print(f"Threshold      : {thr_at_5fpr:.6f}")

# usage with transformers

In [ ]:
# from transformers import (
#     AutoProcessor,
#     AutoModelForCausalLM,
#     BitsAndBytesConfig,
# )
#
# import torch
# import os
# from src.detector.detector_utils import huggingface_config, from_pretrained
#
#
# cache_dir = os.getenv('CACHE_DIR')
# MODEL_ID = "google/gemma-4-E4B-it"
#
# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_quant_type="nf4",
#     bnb_4bit_compute_dtype=torch.bfloat16,
#     bnb_4bit_use_double_quant=True,
# )
#
# processor_kwargs = {"token": huggingface_config["TOKEN"]}
#
# model_kwargs = {
#     #"quantization_config": bnb_config,         # remplace dtype="auto"
#     "device_map": {"": "cuda:0"},
#     "attn_implementation": "sdpa",             # ou "flash_attention_2"
#     "token": huggingface_config["TOKEN"],
# }
#
# processor = from_pretrained(cache_dir, AutoProcessor, MODEL_ID, processor_kwargs)
# model = from_pretrained(cache_dir, AutoModelForCausalLM, MODEL_ID, model_kwargs)
#
# messages = [
#     {"role": "system", "content": prompt_engineering},
#     {"role": "user", "content": "give me a proper review"},
# ]
#
# text = processor.apply_chat_template(
#     messages,
#     tokenize=False,
#     add_generation_prompt=True,
#     enable_thinking=True,
# )
# inputs = processor(text=text, return_tensors="pt").to(model.device)
# input_len = inputs["input_ids"].shape[-1]
#
# with torch.inference_mode():
#     out = model.generate(
#         **inputs,
#         max_new_tokens=2048,
#         do_sample=True,
#         temperature=1.0,
#         top_p=0.95,
#         top_k=64,
#         pad_token_id=processor.tokenizer.eos_token_id
#         #cache_implementation="quantized",
#         #cache_config={"backend": "quanto", "nbits": 4},
#     )
#
# response = processor.decode(out[0][input_len:], skip_special_tokens=False)
# parsed = processor.parse_response(response)
# print(parsed)